In [1]:
from dotenv import load_dotenv
import os

load_dotenv("config.env")

assert os.getenv('OPENAI_API_KEY') is not None
assert os.getenv('TAVILY_API_KEY') is not None

In [2]:
from typing import TypedDict, List, Dict, Any

from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults

from langgraph.graph import StateGraph, END

In [3]:
class HealthBotState(TypedDict):
    topic: str
    search_results: List[Dict[str, Any]]
    formatted_sources: str
    summary: str
    quiz_question: str
    patient_answer: str
    grade: str
    feedback: str
    continue_session: str

In [4]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

tavily_tool = TavilySearchResults(
    max_results=5
)

In [5]:
def format_tavily_results(results: List[Dict[str, Any]]) -> str:
    formatted = []

    for index, result in enumerate(results, start=1):
        title = result.get("title", "Untitled source")
        url = result.get("url", "No URL")
        content = result.get("content", "")

        formatted.append(
            f"[{index}] Title: {title}\n"
            f"URL: {url}\n"
            f"Content: {content}\n"
        )

    return "\n---\n".join(formatted)

In [6]:
def ask_topic(state: HealthBotState) -> HealthBotState:
    topic = input("What health topic or medical condition would you like to learn about? ")

    return {
        "topic": topic
    }

In [7]:
def search_tavily(state: HealthBotState) -> HealthBotState:
    topic = state["topic"]

    query = (
        f"Patient-friendly medical information about {topic}. "
        f"Use reputable medical sources such as CDC, NHS, Mayo Clinic, WHO, "
        f"Cleveland Clinic, MedlinePlus, or major hospital websites."
    )

    results = tavily_tool.invoke({"query": query})
    formatted_sources = format_tavily_results(results)

    return {
        "search_results": results,
        "formatted_sources": formatted_sources
    }

In [8]:
def summarize_results(state: HealthBotState) -> HealthBotState:
    topic = state["topic"]
    sources = state["formatted_sources"]

    prompt = f"""
You are a patient education assistant.

The patient wants to learn about: {topic}

Use the sources below to write a simple, patient-friendly explanation.

Requirements:
- Use clear and simple language.
- Do not give a diagnosis.
- Do not replace professional medical advice.
- Include important symptoms, causes, treatments, or care advice if available.
- Include citations using the source numbers like [1], [2], [3].
- Keep it concise but useful.

Sources:
{sources}
"""

    response = llm.invoke(prompt)

    return {
        "summary": response.content
    }

In [9]:
def show_summary(state: HealthBotState) -> HealthBotState:
    print("\n" + "=" * 80)
    print("HEALTH INFORMATION SUMMARY")
    print("=" * 80)
    print(state["summary"])
    print("=" * 80 + "\n")

    return {
        "summary": state["summary"]
    }

In [10]:
def wait_for_ready(state: HealthBotState) -> HealthBotState:
    input("Press Enter when you are ready for a short comprehension check.")

    return {
        "summary": state["summary"]
    }

In [11]:
def generate_quiz(state: HealthBotState) -> HealthBotState:
    summary = state["summary"]

    prompt = f"""
Create one simple comprehension quiz question based only on the health information summary below.

Requirements:
- Ask only one question.
- The question should check basic understanding.
- Do not ask for obscure details.
- Do not include the answer.

Summary:
{summary}
"""

    response = llm.invoke(prompt)

    return {
        "quiz_question": response.content
    }

In [12]:
def ask_quiz(state: HealthBotState) -> HealthBotState:
    print("\n" + "=" * 80)
    print("COMPREHENSION CHECK")
    print("=" * 80)
    print(state["quiz_question"])
    print("=" * 80 + "\n")

    answer = input("Your answer: ")

    return {
        "patient_answer": answer
    }

In [13]:
def grade_answer(state: HealthBotState) -> HealthBotState:
    summary = state["summary"]
    quiz_question = state["quiz_question"]
    patient_answer = state["patient_answer"]

    prompt = f"""
You are grading a patient's answer to a health education quiz.

Use only the summary below as the source of truth.

Summary:
{summary}

Quiz question:
{quiz_question}

Patient answer:
{patient_answer}

Give:
1. A grade: Correct, Partially Correct, or Incorrect.
2. A short explanation.
3. Relevant citations from the summary, using citation labels like [1], [2], [3].
4. A kind correction if the answer is wrong or incomplete.

Do not introduce new medical facts that are not in the summary.
"""

    response = llm.invoke(prompt)

    return {
        "feedback": response.content
    }

In [14]:
def show_grade(state: HealthBotState) -> HealthBotState:
    print("\n" + "=" * 80)
    print("QUIZ FEEDBACK")
    print("=" * 80)
    print(state["feedback"])
    print("=" * 80 + "\n")

    return {
        "feedback": state["feedback"]
    }

In [15]:
def ask_continue(state: HealthBotState) -> HealthBotState:
    choice = input("Would you like to learn about another topic? Type 'yes' or 'no': ")

    return {
        "continue_session": choice.strip().lower()
    }

In [16]:
def route_continue(state: HealthBotState) -> str:
    if state["continue_session"] in ["yes", "y"]:
        return "reset_state"
    else:
        return END

In [17]:
def reset_state(state: HealthBotState) -> HealthBotState:
    return {
        "topic": "",
        "search_results": [],
        "formatted_sources": "",
        "summary": "",
        "quiz_question": "",
        "patient_answer": "",
        "grade": "",
        "feedback": "",
        "continue_session": ""
    }

In [18]:
workflow = StateGraph(HealthBotState)

workflow.add_node("ask_topic", ask_topic)
workflow.add_node("search_tavily", search_tavily)
workflow.add_node("summarize_results", summarize_results)
workflow.add_node("show_summary", show_summary)
workflow.add_node("wait_for_ready", wait_for_ready)
workflow.add_node("generate_quiz", generate_quiz)
workflow.add_node("ask_quiz", ask_quiz)
workflow.add_node("grade_answer", grade_answer)
workflow.add_node("show_grade", show_grade)
workflow.add_node("ask_continue", ask_continue)
workflow.add_node("reset_state", reset_state)

In [19]:
workflow.set_entry_point("ask_topic")

workflow.add_edge("ask_topic", "search_tavily")
workflow.add_edge("search_tavily", "summarize_results")
workflow.add_edge("summarize_results", "show_summary")
workflow.add_edge("show_summary", "wait_for_ready")
workflow.add_edge("wait_for_ready", "generate_quiz")
workflow.add_edge("generate_quiz", "ask_quiz")
workflow.add_edge("ask_quiz", "grade_answer")
workflow.add_edge("grade_answer", "show_grade")
workflow.add_edge("show_grade", "ask_continue")

workflow.add_conditional_edges(
    "ask_continue",
    route_continue
)

workflow.add_edge("reset_state", "ask_topic")

In [20]:
healthbot_app = workflow.compile()

In [ ]:
initial_state = {
    "topic": "",
    "search_results": [],
    "formatted_sources": "",
    "summary": "",
    "quiz_question": "",
    "patient_answer": "",
    "grade": "",
    "feedback": "",
    "continue_session": ""
}

healthbot_app.invoke(initial_state)

What health topic or medical condition would you like to learn about?  cancer



HEALTH INFORMATION SUMMARY
**Understanding Cancer: A Simple Guide**

Cancer is a term used for a group of diseases where abnormal cells grow uncontrollably in the body. These cells can form lumps called tumors and may spread to other parts of the body, a process known as metastasis [2][3]. 

### Common Symptoms
Symptoms of cancer can vary widely depending on the type and stage of the disease. Some general signs to watch for include:
- Unexplained weight loss
- Fatigue that doesn't go away
- Pain that doesn't have a clear cause
- Changes in skin, such as new moles or changes in existing ones
- Persistent cough or difficulty breathing
- Changes in bowel or bladder habits

If you notice any unusual changes in your body that last more than two weeks, it's important to talk to a healthcare provider [3][4].

### Causes of Cancer
Cancer can be caused by a mix of genetic factors and environmental influences. Some common risk factors include:
- Tobacco use
- Excessive alcohol consumption
- Obe

Press Enter when you are ready for a short comprehension check. 



COMPREHENSION CHECK
What is one common symptom of cancer mentioned in the summary?



Your answer:  chemo



QUIZ FEEDBACK
1. Grade: Incorrect

2. Explanation: The patient's answer "chemo" refers to chemotherapy, which is a treatment option for cancer, not a symptom of cancer. The question specifically asks for a common symptom of cancer.

3. Relevant citations: The summary lists common symptoms of cancer, including "unexplained weight loss," "fatigue that doesn't go away," "pain that doesn't have a clear cause," "changes in skin," "persistent cough or difficulty breathing," and "changes in bowel or bladder habits" [2][3].

4. Correction: A common symptom of cancer could be "unexplained weight loss" or "fatigue that doesn't go away." Please refer to the summary for a list of symptoms.



Would you like to learn about another topic? Type 'yes' or 'no':  yes
